In [34]:
# ABC = Abstract Base Class. We use it to define a common structure for Runnables.
# abstractmethod means every Runnable must provide its own invoke() method.
from abc import ABC, abstractmethod


In [35]:
# This is the basic interface that every Runnable in our example must follow.
# If a class is a Runnable, we expect it to have an invoke() method.
class Runnable(ABC):

    @abstractmethod
    def invoke(input_data):
        pass


In [36]:
# We use random only to simulate different LLM responses.
import random

# This is a fake/mock LLM used only to understand the Runnable concept.
# In real LangChain, this could be a ChatModel or another LLM implementation.
class NakliLLM(Runnable):
    def __init__(self):
        print("LLM Created..")

    # invoke() follows the common Runnable interface.
    def invoke(self, prompt):
        # A real LLM would generate a response from the prompt.
        # Here we simply return one predefined response.
        response_list = [
                    "Delhi is the capital of India",
                    "Ms. Cailin is my favourite character",
                    "Satoru Gojo is the Honored One"
                ]

        # We return a dictionary so the next Runnable can receive structured data.
        return {"response": random.choice(response_list)}

    # predict() is another way to get a response in this mock example.
    # It is included for comparison, but our chain uses invoke().
    def predict(self, prompt):
        response_list = [
            "Delhi is the capital of India",
            "Ms. Cailin is my favourite character",
            "Satoru Gojo is the Honored One"
        ]

        return {"response": random.choice(response_list)}



In [37]:
# Create an instance of our fake LLM.
# This object can now be used as a Runnable.
llm = NakliLLM()


LLM Created..


In [38]:
# This is a simplified version of LangChain's PromptTemplate.
# It stores a reusable prompt containing placeholders such as {topic}.
class NakliPromptTemplate(Runnable):
    def __init__(self, template, input_variables):
        self.template = template
        self.input_variables = input_variables

    # invoke() receives values and fills the placeholders in the template.
    def invoke(self, input_dict):
        return self.template.format(**input_dict)

    # format() does the same formatting directly.
    # We keep it here to show that formatting can also be called explicitly.
    def format(self, input_dict):
        return self.template.format(**input_dict)



In [39]:
# Create a reusable prompt.
# {topic} is a placeholder that will be replaced at runtime.
template = NakliPromptTemplate(
    template = "What is {topic} topic is actually?",
    input_variables = ["topic"]
)


In [40]:
# Pass the actual value for {topic}.
# invoke() replaces {topic} with "ML" and returns the final prompt string.
template.invoke({"topic": "ML"})


'What is ML topic is actually?'

#### Standardizing the components

Each component follows the same `Runnable` interface: it receives input through `invoke()` and returns output.

**Simple idea:** because every component behaves in the same basic way, we can connect them together.


In [41]:
# This Runnable converts the LLM's dictionary response into plain text.
# Example: {"response": "..."}  ->  "..."
class NakliStrOutputParser(Runnable):
    def __init__(self):
        pass

    def invoke(self, input_dict):
        return input_dict["response"]


In [42]:
# RunnableConnector is our simple version of a chain.
# It connects multiple Runnables and executes them one after another.
class RunnableConnector(Runnable):
    def __init__(self, runnable_list):
        self.runnable_list = runnable_list

    def invoke(self, input_data):
        # The output of one Runnable becomes the input of the next Runnable.
        for runnable in self.runnable_list:
            input_data = runnable.invoke(input_data)

        return input_data


In [43]:
# First Runnable: create a prompt about the given DongHua name.
prompt = NakliPromptTemplate(
    template = "Write a detailed report on DongHua {name}",
    input_variables = ["name"]
)


In [51]:
# Second Runnable: our fake LLM.
llm = NakliLLM()


LLM Created..


In [52]:
# Connect PromptTemplate -> LLM.
# Data flows from left to right:
# input dictionary -> formatted prompt -> LLM response dictionary
chain = RunnableConnector([prompt, llm,])


In [53]:
# Start the chain with the value required by the prompt.
# The chain automatically passes each output to the next Runnable.
chain.invoke({"name": "Battle Through The Heavens"})


{'response': 'Satoru Gojo is the Honored One'}

## Combining 3 Runnables

Now we connect **PromptTemplate → LLM → OutputParser**.

The important idea is that the **output of one Runnable becomes the input of the next Runnable**.


In [54]:
# Create the third Runnable: convert the LLM dictionary into a string.
parser = NakliStrOutputParser()


In [ ]:
# Now the complete flow is:
# PromptTemplate -> LLM -> OutputParser
# Each component does one small job.
chain = RunnableConnector([prompt, llm, parser])


In [56]:
# Run the complete three-step chain.
# Final result is plain text because the parser runs after the LLM.
chain.invoke({"name": "Battle Through The Heavens"})


'Ms. Cailin is my favourite character'

## Creating a small application using 2 Chains (Runnables)

Here we build two smaller workflows and then combine them:

1. **Chain 1:** generate a joke.
2. **Chain 2:** take that joke and explain it.
3. **Final chain:** connect Chain 1 → Chain 2.

This demonstrates how larger workflows can be built by composing smaller reusable components.


In [57]:
# First prompt: ask the LLM to generate a joke.
template1 = NakliPromptTemplate(
    template = "Write a joke about {topic}",
    input_variables = ["topic"]
)


In [58]:
# Second prompt: take the first LLM response and ask the LLM to explain it.
# {response} will receive the output produced by chain1.
template2 = NakliPromptTemplate(
    template = "Explain the follwing text: {response}",
    input_variables = ["response"]
)


In [59]:
# The same LLM can be reused by multiple chains.
llm = NakliLLM()


LLM Created..


In [60]:
# Parser for the final LLM response.
parser = NakliStrOutputParser()


In [66]:
# Chain 1:
# topic -> joke prompt -> LLM response
chain1 = RunnableConnector([template1, llm])


In [67]:
# Chain 2:
# chain1's response -> explanation prompt -> LLM -> plain text
chain2 = RunnableConnector([template2, llm, parser])


In [68]:
# Combine the two chains into one larger workflow.
# final_chain runs chain1 first, then passes its output to chain2.
final_chain = RunnableConnector([chain1, chain2])


In [69]:
# Start the complete application with the initial topic.
# Data flows through both chains automatically.
final_chain.invoke({"topic": "Joke"})


'Satoru Gojo is the Honored One'